# autoresearch: Pallas ReLU(XW+b) on TPU v6e

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jake-Song/tensor2silicon/blob/main/notebooks/autoresearch_tpu_v6e.ipynb)

Claude Code를 Colab 터미널에서 실행해 `autoresearch/tpu/kernel.py` 하나만 고치면서 `Y = ReLU(XW + b)` 커널을 스스로 최적화하게 한다. 한 번의 실험은 **가설 → kernel.py 수정 → commit → `bench.py` → 빨라졌으면 유지, 아니면 `git reset`** 이다. 규칙은 `autoresearch/tpu/program.md`, 채점은 고정된 `autoresearch/tpu/bench.py`가 맡는다.

**런타임 설정**: 메뉴 `런타임 → 런타임 유형 변경 → v6e-1 TPU`.

이 노트북은 준비(1~3), 에이전트 시작 안내(4), 진행 상황 보기(5), 결과 저장(6)을 맡는다. 에이전트 자체는 터미널에서 돈다.

## 0. 환경 확인

`kind`가 `TPU v6 lite`여야 한다. v5e 등에서도 돌지만 program.md의 하드웨어 수치와 `pct_peak`가 맞지 않는다.

In [ ]:
# 노트북 커널에서는 jax를 import하지 않는다. TPU는 한 프로세스만 잡을 수 있어서,
# 여기서 잡으면 bench.py와 터미널의 에이전트가 TPU를 쓰지 못한다. 그래서 확인도 별도 프로세스로 한다.
!python -c "import jax; d = jax.devices()[0]; print('jax', jax.__version__, '| platform', d.platform, '| kind', d.device_kind)"

### 0-1. Pallas가 실제 TPU에서 컴파일되는지 확인

Colab 기본 이미지의 jax와 libtpu 버전이 어긋나면 Mosaic 컴파일이 실패한다. 2026-09 기준 jax 0.7.2 + libtpu 0.0.21.1에서 `Unsupported version: expected <= 7 but got 8`이 났다. 그대로 두면 에이전트가 커널 문제로 착각하므로 시작 전에 확인한다.
검사가 실패하면 `jax[tpu]`를 최신으로 올린다. 확인한 조합은 jax 0.11.2 + libtpu 0.0.48이다. `bench.py`와 에이전트는 매번 새 프로세스로 실행되므로 런타임을 다시 시작하지 않아도 된다.

In [ ]:
import subprocess, sys

SMOKE = '''
import jax, jax.numpy as jnp
from jax.experimental import pallas as pl
def add_one(x_ref, o_ref):
    o_ref[...] = x_ref[...] + 1.0
out = pl.pallas_call(add_one, out_shape=jax.ShapeDtypeStruct((8, 128), jnp.float32))(jnp.zeros((8, 128)))
assert bool((out == 1.0).all())
print("Pallas on TPU (Mosaic): OK | jax", jax.__version__)
'''

def smoke():
    r = subprocess.run([sys.executable, "-c", SMOKE], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip().splitlines()[-1][:200])
    return r.returncode == 0

if not smoke():
    print("-> jax/libtpu 버전 불일치로 보고 jax[tpu]를 올립니다")
    !pip install -q -U "jax[tpu]"
    assert smoke(), "업그레이드 후에도 실패: 위 오류를 확인하세요"
!pip list 2>/dev/null | grep -E "^(jax|jaxlib|libtpu) "

## 1. 저장소 받기

`/content/tensor2silicon`에 clone한다. 에이전트가 실험마다 commit하므로 git 사용자 정보도 넣는다.

In [ ]:
import os
REPO = "/content/tensor2silicon"
WORKDIR = f"{REPO}/autoresearch/tpu"
if not os.path.exists(REPO):
    !git clone https://github.com/Jake-Song/tensor2silicon.git {REPO}
else:
    !git -C {REPO} pull --ff-only
!git -C {REPO} config user.name "autoresearch"
!git -C {REPO} config user.email "autoresearch@localhost"
%cd {WORKDIR}
!ls -a

## 2. 베이스라인 실행

에이전트가 이길 기준점이다. `status: ok`, `correct: True`가 나와야 한다. 일부러 튜닝하지 않은 커널이라 `speedup`(XLA 대비)은 1보다 한참 작다.

In [ ]:
!python bench.py 2>&1 | tail -n 12

## 3. Claude Code 설치

네이티브 설치 스크립트는 `~/.local/bin/claude`에 설치한다.

In [ ]:
!curl -fsSL https://claude.ai/install.sh | bash
!~/.local/bin/claude --version

### 3-1. (선택) API 키로 인증

Claude 구독 계정이면 건너뛰고, 터미널에서 `claude`를 처음 실행할 때 나오는 로그인 URL로 인증한다.
API 키를 쓰려면 Colab 왼쪽 🔑 **Secrets**에 `ANTHROPIC_API_KEY`를 넣고 아래 셀을 실행한다. 터미널은 노트북의 secret을 못 보므로 `~/.bashrc`에 적는다. 이 VM은 세션이 끝나면 사라진다.

In [ ]:
from google.colab import userdata
key = userdata.get("ANTHROPIC_API_KEY")
with open(os.path.expanduser("~/.bashrc"), "a") as f:
    f.write(f'\nexport ANTHROPIC_API_KEY="{key}"\n')
print("~/.bashrc에 ANTHROPIC_API_KEY를 추가했습니다. 새 터미널부터 적용됩니다.")

## 4. 에이전트 시작 (터미널)

Colab 왼쪽 아래 **터미널** 아이콘을 눌러 연 뒤:

```bash
cd /content/tensor2silicon/autoresearch/tpu
export PATH="$HOME/.local/bin:$PATH"
claude
```

Claude Code에 이렇게 입력한다 (태그는 날짜 등 아무 이름):

```
Read program.md and kick off a new experiment run. Tag: sep23. Stop after 30 experiments.
```

- 에이전트는 `autoresearch/<tag>-tpu` 브랜치를 만들고, 베이스라인을 돌린 뒤 한 번 확인을 받고 루프에 들어간다.
- `.claude/settings.json`이 `bench.py` 실행, git, `kernel.py`/`results.tsv` 편집만 허용하고 `bench.py`·`program.md` 편집은 막는다. 허용 목록 밖의 명령은 승인을 묻는다. 버려도 되는 VM이니 완전 무인으로 돌리려면 `claude --dangerously-skip-permissions`로 시작해도 된다.
- 실험 1회에 1~2분. 30회면 대략 1시간이다. TPU 런타임도 컴퓨팅 단위를 쓰므로 실험 수 상한을 정해 둔다.
- 브라우저 탭을 닫으면 런타임이 끊길 수 있다. 로컬의 Colab CLI(`uv tool install google-colab-cli`)로 `colab new --tpu v6e1` → `colab console`을 쓰면 tmux 셸과 keep-alive가 붙어 탭 없이도 돈다.

## 5. 진행 상황 보기

에이전트가 도는 동안 아무 때나 다시 실행한다. `results.tsv`는 에이전트가 실험마다 한 줄씩 추가한다.
`report.md`에는 성공뿐 아니라 느려지거나 틀렸거나 실패한 시도도 가설·결과·교훈과 함께 남는다. 되돌린 커밋의 코드는 브랜치에서 사라지므로 이 문서가 유일한 기록이다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("results.tsv", sep="\t")
display(df)

ok = df[df.time_ms > 0].copy()
ok["best_ms"] = ok.time_ms.cummin()
line = !grep '^baseline_ms:' run.log
baseline_ms = float(line[0].split()[1]) if line else float("nan")   # vendor 기준선 (가장 최근 run)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(ok.index, ok.time_ms, s=36, color="#b3b2ad", zorder=2, label="experiment")
ax.step(ok.index, ok.best_ms, where="post", lw=2, color="#2a78d6", zorder=3, label="best so far")
if baseline_ms == baseline_ms:
    ax.axhline(baseline_ms, ls="--", lw=1, color="#52514e")
    ax.annotate("XLA", (ok.index.max(), baseline_ms), xytext=(0, 4), textcoords="offset points",
                ha="right", va="bottom", color="#52514e", fontsize=9)
ax.set_xlabel("experiment #")
ax.set_ylabel("time_ms (lower is better)")
ax.set_ylim(bottom=0)
ax.grid(axis="y", color="#e6e5e1", lw=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper right")
ax.set_title(f"Pallas ReLU(XW+b) on TPU v6e: best {ok.best_ms.iloc[-1]:.3f} ms "
             f"(baseline kernel {ok.time_ms.iloc[0]:.3f} ms)", loc="left", fontsize=11)
plt.show()

In [ ]:
from IPython.display import Markdown
display(Markdown(open("report.md").read()) if os.path.exists("report.md") else "report.md가 아직 없습니다")

## 6. 결과를 Google Drive에 저장

런타임이 끝나면 VM의 파일과 git 브랜치가 사라진다. 브랜치 전체를 `git bundle`로, 로그·`report.md`·최종 커널을 파일로 복사한다.
나중에 로컬에서 `git fetch <번들 경로> <브랜치>:<브랜치>`로 불러올 수 있다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

branch = !git rev-parse --abbrev-ref HEAD
branch = branch[0]
out = f"/content/drive/MyDrive/tensor2silicon-autoresearch/{branch.replace('/', '_')}"
os.makedirs(out, exist_ok=True)
!git bundle create {out}/branch.bundle {branch}
!cp results.tsv run.log report.md kernel.py {out}/
!ls -l {out}